<a href="https://colab.research.google.com/github/JudithJacquet/TFG-Pronostico-Picos-SIN/blob/prueba_modelos/Primer_Modelo_Maximo_Diario.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Primer modelo: pronóstico del máximo diario de demanda del SIN

## Objetivo

Aprender paso a paso el proceso completo de construcción, entrenamiento y evaluación de un modelo de pronóstico.

Se utilizará inicialmente una regresión lineal sencilla para predecir directamente la demanda máxima del día siguiente.

El cuaderno tendrá fines de aprendizaje. Sus resultados no se considerarán definitivos para la tesis sin la correspondiente validación metodológica.


In [ ]:
import pandas as pd

from google.colab import files

# Seleccionar el archivo desde nuestra computadora
uploaded = files.upload()

# Obtener el nombre del archivo seleccionado
nombre_archivo = list(uploaded.keys())[0]

# Leer el CSV
df = pd.read_csv(nombre_archivo)

# Comprobar la carga
print("Archivo:", nombre_archivo)
print("Dimensiones:", df.shape)

display(df.head())

Saving complete-dataset.csv to complete-dataset (1).csv
Archivo: complete-dataset (1).csv
Dimensiones: (131481, 9)


,DATETIME,SIN,T02M,RH2M,PRSS,TPP6,U10M,V10M,ISHOLIDAY
0,2008-01-01 00:00:00-03:00,1060.5,24.463579,96.798320,999.585909,0.000629,-1.613880,-1.142420,1.0
1,2008-01-01 01:00:00-03:00,990.0,24.314531,96.065988,999.059392,0.000419,-1.624135,-1.256589,1.0
2,2008-01-01 02:00:00-03:00,957.5,24.165483,95.333657,998.532875,0.000210,-1.634390,-1.370759,1.0
3,2008-01-01 03:00:00-03:00,919.5,24.016435,94.601325,998.006359,0.000000,-1.644645,-1.484929,1.0
4,2008-01-01 04:00:00-03:00,890.8,23.667572,94.257455,998.982867,0.000000,-1.273453,-1.629541,1.0


PREPROCESAMIENTO

In [ ]:
# Convertir las fechas originales a fechas interpretables por Pandas
df["DATETIME_UTC"] = pd.to_datetime(df["DATETIME"], utc=True)

# Convertirlas a la zona horaria de Paraguay
df["DATETIME_LOCAL"] = (df["DATETIME_UTC"].dt.tz_convert("America/Asuncion"))

# Mostrar las primeras fechas para comprobar el resultado
display(df[["DATETIME", "DATETIME_UTC", "DATETIME_LOCAL"]].head())

,DATETIME,DATETIME_UTC,DATETIME_LOCAL
0,2008-01-01 00:00:00-03:00,2008-01-01 03:00:00+00:00,2008-01-01 00:00:00-03:00
1,2008-01-01 01:00:00-03:00,2008-01-01 04:00:00+00:00,2008-01-01 01:00:00-03:00
2,2008-01-01 02:00:00-03:00,2008-01-01 05:00:00+00:00,2008-01-01 02:00:00-03:00
3,2008-01-01 03:00:00-03:00,2008-01-01 06:00:00+00:00,2008-01-01 03:00:00-03:00
4,2008-01-01 04:00:00-03:00,2008-01-01 07:00:00+00:00,2008-01-01 04:00:00-03:00


In [ ]:
# Contar las filas completamente duplicadas
duplicados = df.duplicated().sum()

print("Cantidad de registros duplicados:", duplicados)

# Mostrar algunos de los registros involucrados
display(df.loc[df.duplicated(keep=False), ["DATETIME", "SIN"]].head(10))

Cantidad de registros duplicados: 15


,DATETIME,SIN
7007,2008-10-19 01:00:00-03:00,795.2
7008,2008-10-19 01:00:00-03:00,795.2
15742,2009-10-18 01:00:00-03:00,866.6
15743,2009-10-18 01:00:00-03:00,866.6
24141,2010-10-03 01:00:00-03:00,855.9
24142,2010-10-03 01:00:00-03:00,855.9
32876,2011-10-02 01:00:00-03:00,902.9
32877,2011-10-02 01:00:00-03:00,902.9
41779,2012-10-07 01:00:00-03:00,1582.0
41780,2012-10-07 01:00:00-03:00,1582.0


In [ ]:
# Eliminar registros completamente duplicados
df = df.drop_duplicates()

# Ordenar cronológicamente por fecha UTC
df = df.sort_values("DATETIME_UTC")

# Reiniciar la numeración de las filas
df = df.reset_index(drop=True)

# Comprobar el resultado
print("Dimensiones después de eliminar duplicados:", df.shape)
print("Duplicados restantes:", df.duplicated().sum())

Dimensiones después de eliminar duplicados: (131466, 11)
Duplicados restantes: 0


In [ ]:
# Extraer la fecha local de Paraguay
df["FECHA"] = df["DATETIME_LOCAL"].dt.date

# Calcular el máximo diario y contar las observaciones
df_diario = (
    df.groupby("FECHA")
    .agg(
        SIN_max=("SIN", "max"),
        N_REGISTROS=("SIN", "count")
    )
    .reset_index()
)

# Mostrar el resultado
display(df_diario.head(10))

print("Cantidad de días:", len(df_diario))


,FECHA,SIN_max,N_REGISTROS
0,2008-01-01,1251.0,24
1,2008-01-02,1440.6,24
2,2008-01-03,1458.0,24
3,2008-01-04,1405.0,24
4,2008-01-05,1379.0,24
5,2008-01-06,1372.6,24
6,2008-01-07,1456.0,24
7,2008-01-08,1467.0,24
8,2008-01-09,1490.0,24
9,2008-01-10,1526.7,24


Cantidad de días: 5479


In [ ]:
# Ordenar la tabla diaria por fecha
df_diario = df_diario.sort_values("FECHA").reset_index(drop=True)

# Crear la variable de entrada: máximo del día anterior
df_diario["MAX_AYER"] = df_diario["SIN_max"].shift(1)

# Mostrar el resultado
display(df_diario[ ["FECHA", "MAX_AYER", "SIN_max"]].head(10))

,FECHA,MAX_AYER,SIN_max
0,2008-01-01,NaN,1251.0
1,2008-01-02,1251.0,1440.6
2,2008-01-03,1440.6,1458.0
3,2008-01-04,1458.0,1405.0
4,2008-01-05,1405.0,1379.0
5,2008-01-06,1379.0,1372.6
6,2008-01-07,1372.6,1456.0
7,2008-01-08,1456.0,1467.0
8,2008-01-09,1467.0,1490.0
9,2008-01-10,1490.0,1526.7


In [ ]:
# Asegurar que FECHA tenga formato de fecha
df_diario["FECHA"] = pd.to_datetime(df_diario["FECHA"])

# Verificar que cada fila corresponda al día siguiente
es_consecutivo = (
    df_diario["FECHA"].diff() == pd.Timedelta(days=1)
)

# Invalidar MAX_AYER si la fecha anterior no es consecutiva
df_diario.loc[~es_consecutivo, "MAX_AYER"] = float("nan")

# Excluir filas sin entrada u objetivo
datos_modelo = df_diario.dropna(
    subset=["MAX_AYER", "SIN_max"]
)

# Separar entradas y salidas
X = datos_modelo[["MAX_AYER"]]
y = datos_modelo["SIN_max"]

print("Dimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)

display(X.head())
display(y.head())

Dimensiones de X: (5478, 1)
Dimensiones de y: (5478,)


,MAX_AYER
1,1251.0
2,1440.6
3,1458.0
4,1405.0
5,1379.0


,SIN_max
1,1440.6
2,1458.0
3,1405.0
4,1379.0
5,1372.6


In [ ]:
print("Filas en df_diario:", len(df_diario))
print("Filas en datos_modelo:", len(datos_modelo))

print("\nPrimera fila de df_diario:")
display(df_diario.head(1))

print("\nPrimera fila de datos_modelo:")
display(datos_modelo.head(1))


Filas en df_diario: 5479
Filas en datos_modelo: 5478

Primera fila de df_diario:


,FECHA,SIN_max,N_REGISTROS,MAX_AYER
0,2008-01-01,1251.0,24,NaN



Primera fila de datos_modelo:


,FECHA,SIN_max,N_REGISTROS,MAX_AYER
1,2008-01-02,1440.6,24,1251.0


In [ ]:
# Ordenar los ejemplos cronológicamente
datos_modelo = datos_modelo.sort_values("FECHA")

# Determinar el punto de separación
n_train = int(len(datos_modelo) * 0.8)

# Separar entrenamiento y prueba
train = datos_modelo.iloc[:n_train]
test = datos_modelo.iloc[n_train:]

# Aplicar el pronóstico naive sobre los datos de prueba
pred_naive = test["MAX_AYER"]

# Mostrar algunos resultados
resultados_naive = test[["FECHA", "SIN_max"]].copy()

resultados_naive["PREDICCION"] = pred_naive

print("Ejemplos de entrenamiento:", len(train))
print("Ejemplos de prueba:", len(test))

display(resultados_naive.head(10))

Ejemplos de entrenamiento: 4382
Ejemplos de prueba: 1096


,FECHA,SIN_max,PREDICCION
4383,2020-01-01,2492.0,3010.0
4384,2020-01-02,2762.0,2492.0
4385,2020-01-03,2749.0,2762.0
4386,2020-01-04,2597.0,2749.0
4387,2020-01-05,2920.5,2597.0
4388,2020-01-06,3088.0,2920.5
4389,2020-01-07,2835.0,3088.0
4390,2020-01-08,3145.0,2835.0
4391,2020-01-09,3219.0,3145.0
4392,2020-01-10,3195.0,3219.0


In [ ]:
# Calcular el error de cada pronóstico
resultados_naive["ERROR"] = (
    resultados_naive["SIN_max"] - resultados_naive["PREDICCION"]
)

# Calcular el error absoluto
resultados_naive["ERROR_ABSOLUTO"] = (
    resultados_naive["ERROR"].abs()
)

# Mostrar los primeros resultados
display(resultados_naive.head(10))


,FECHA,SIN_max,PREDICCION,ERROR,ERROR_ABSOLUTO
4383,2020-01-01,2492.0,3010.0,-518.0,518.0
4384,2020-01-02,2762.0,2492.0,270.0,270.0
4385,2020-01-03,2749.0,2762.0,-13.0,13.0
4386,2020-01-04,2597.0,2749.0,-152.0,152.0
4387,2020-01-05,2920.5,2597.0,323.5,323.5
4388,2020-01-06,3088.0,2920.5,167.5,167.5
4389,2020-01-07,2835.0,3088.0,-253.0,253.0
4390,2020-01-08,3145.0,2835.0,310.0,310.0
4391,2020-01-09,3219.0,3145.0,74.0,74.0
4392,2020-01-10,3195.0,3219.0,-24.0,24.0


In [ ]:
# Calcular el error absoluto medio
mae_naive = resultados_naive["ERROR_ABSOLUTO"].mean()

# Mostrar el resultado
print(f"MAE del modelo naive: {mae_naive:.2f} MW")

MAE del modelo naive: 188.60 MW


In [ ]:
# Máximo diario promedio durante el período de prueba
demanda_promedio = test["SIN_max"].mean()

# MAE en relación con ese promedio
nmae = (mae_naive / demanda_promedio) * 100

print(f"Máximo diario promedio: {demanda_promedio:.2f} MW")
print(f"MAE: {mae_naive:.2f} MW")
print(f"MAE relativo al promedio: {nmae:.2f} %")

Máximo diario promedio: 2662.36 MW
MAE: 188.60 MW
MAE relativo al promedio: 7.08 %


#REGRESION LINEAL


In [ ]:
# Importar el modelo de regresión lineal
from sklearn.linear_model import LinearRegression

# Separar las entradas y salidas de entrenamiento
X_train = train[["MAX_AYER"]]
y_train = train["SIN_max"]

# Crear el modelo
modelo_lineal = LinearRegression()

# Entrenar el modelo
modelo_lineal.fit(X_train, y_train)

# Mostrar los parámetros aprendidos
print("Intercepto (a):", modelo_lineal.intercept_)
print("Pendiente (b):", modelo_lineal.coef_[0])

Intercepto (a): 176.78154439270975
Pendiente (b): 0.9080544667000815


In [ ]:
# Preparar las entradas del conjunto de prueba
X_test = test[["MAX_AYER"]]

# Generar las predicciones de la regresión lineal
pred_lineal = modelo_lineal.predict(X_test)

# Construir una tabla para comparar los pronósticos
resultados_lineal = test[["FECHA", "SIN_max"]].copy()

resultados_lineal["PRED_NAIVE"] = test["MAX_AYER"]

resultados_lineal["PRED_LINEAL"] = pred_lineal

# Mostrar los primeros resultados
display(resultados_lineal.head(10))

,FECHA,SIN_max,PRED_NAIVE,PRED_LINEAL
4383,2020-01-01,2492.0,3010.0,2910.025489
4384,2020-01-02,2762.0,2492.0,2439.653275
4385,2020-01-03,2749.0,2762.0,2684.827981
4386,2020-01-04,2597.0,2749.0,2673.023273
4387,2020-01-05,2920.5,2597.0,2534.998994
4388,2020-01-06,3088.0,2920.5,2828.754614
4389,2020-01-07,2835.0,3088.0,2980.853738
4390,2020-01-08,3145.0,2835.0,2751.115957
4391,2020-01-09,3219.0,3145.0,3032.612842
4392,2020-01-10,3195.0,3219.0,3099.808873


In [ ]:
# Importar la métrica MAE
from sklearn.metrics import mean_absolute_error

# Valores reales del conjunto de prueba
y_test = test["SIN_max"]

# Calcular el MAE de cada modelo
mae_naive = mean_absolute_error(
    y_test,
    resultados_lineal["PRED_NAIVE"]
)

mae_lineal = mean_absolute_error(
    y_test,
    resultados_lineal["PRED_LINEAL"]
)

# Mostrar los resultados
print(f"MAE naive: {mae_naive:.2f} MW")
print(f"MAE regresión lineal: {mae_lineal:.2f} MW")

MAE naive: 188.60 MW
MAE regresión lineal: 195.55 MW


In [ ]:
# Predicciones de la regresión lineal sobre entrenamiento
pred_lineal_train = modelo_lineal.predict(X_train)

# Predicciones naive sobre entrenamiento
pred_naive_train = train["MAX_AYER"]

# Calcular los errores en entrenamiento
mae_naive_train = mean_absolute_error(
    y_train,
    pred_naive_train
)

mae_lineal_train = mean_absolute_error(
    y_train,
    pred_lineal_train
)

# Mostrar la comparación completa
print("=== ENTRENAMIENTO ===")
print(f"MAE naive: {mae_naive_train:.2f} MW")
print(f"MAE lineal: {mae_lineal_train:.2f} MW")

print("\n=== PRUEBA ===")
print(f"MAE naive: {mae_naive:.2f} MW")
print(f"MAE lineal: {mae_lineal:.2f} MW")

=== ENTRENAMIENTO ===
MAE naive: 129.39 MW
MAE lineal: 129.11 MW

=== PRUEBA ===
MAE naive: 188.60 MW
MAE lineal: 195.55 MW
